# Per-soliton error breakdown at N=12

**What this notebook does:**
1. Pipeline sanity check.
2. Train N=12 at **width=50** (the failing baseline) — 15k Adam + 2k LBFGS.
3. Train N=12 at **width=100** (should rescue) — **25k Adam** + 2k LBFGS.
   More Adam iterations because w=100 has 4x more parameters and needs longer
   to find a good basin before LBFGS can finish the job.
4. Per-soliton L² in a ±3-unit window around each soliton's moving centre.
5. Bar charts + annotated heatmaps + CSV.

**Approx runtime:** ~4.5 h on Kaggle T4 (50 min for w=50, ~2.5 h for w=100 at 25k, rest analysis).

In [ ]:
import glob, sys, os, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

# find kdv_core.py wherever kaggle mounted it
matches = glob.glob('/kaggle/input/**/kdv_core.py', recursive=True)
if not matches:
    raise FileNotFoundError("kdv_core.py not found. Attach the kdv-core dataset.")
sys.path.insert(0, str(Path(matches[0]).parent))
print(f"kdv_core found at: {matches[0]}")

import kdv_core as K
print(f"device = {K.DEVICE}")
print(f"numpy  = {np.__version__}")
print(f"torch  = {torch.__version__}")

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)
print(f"output = {OUT}")


In [ ]:
K.quick_sanity_check(seed=99, adam_iter=500)


### Step 1 — Train N=12 at width=50 (failing baseline, 15k Adam)

In [ ]:
N    = 12
SEED = 99
cfg  = K.make_config(N)
K.print_config(cfg)

ckpt_w50 = OUT / "checkpoint_N12_w50_persoliton.pt"

if ckpt_w50.exists():
    print("[resume] w=50 checkpoint found, skipping training.")
    ck50      = K.load_checkpoint(ckpt_w50)
    model_w50 = K.model_from_checkpoint(ck50)
    hist_w50  = ck50["history"]
else:
    print("=" * 65)
    print("TRAIN N=12  width=50  15k Adam + 2k LBFGS")
    print("=" * 65)
    K.set_seed(SEED)
    batch = K.build_data(cfg, seed=SEED)
    model_w50 = K.PINN(width=50).to(K.DEVICE)
    print(f"  params = {model_w50.n_params():,}")
    hist_w50 = K.train(model_w50, batch, adam_iter=15000, lbfgs_iter=2000)
    K.save_checkpoint(ckpt_w50, model_w50, hist_w50, cfg,
                      extras=dict(width=50, N=N, seed=SEED, adam_iter=15000))
    print(f"  saved {ckpt_w50}")

print(f"\nw=50  Adam L2={hist_w50['adam_l2']:.4f}%  Final L2={hist_w50['lbfgs_l2']:.4f}%")
print(f"Reference: ~42.15% final  (expected to fail)")


### Step 2 — Train N=12 at width=100 (rescued, 25k Adam)

In [ ]:
ckpt_w100 = OUT / "checkpoint_N12_w100_persoliton.pt"

if ckpt_w100.exists():
    print("[resume] w=100 checkpoint found, skipping training.")
    ck100      = K.load_checkpoint(ckpt_w100)
    model_w100 = K.model_from_checkpoint(ck100)
    hist_w100  = ck100["history"]
else:
    print("=" * 65)
    print("TRAIN N=12  width=100  25k Adam + 2k LBFGS")
    print("=" * 65)
    K.set_seed(SEED)
    batch = K.build_data(cfg, seed=SEED)
    model_w100 = K.PINN(width=100).to(K.DEVICE)
    print(f"  params = {model_w100.n_params():,}")
    hist_w100 = K.train(model_w100, batch, adam_iter=25000, lbfgs_iter=2000)
    K.save_checkpoint(ckpt_w100, model_w100, hist_w100, cfg,
                      extras=dict(width=100, N=N, seed=SEED, adam_iter=25000))
    print(f"  saved {ckpt_w100}")

print(f"\nw=100  Adam L2={hist_w100['adam_l2']:.4f}%  Final L2={hist_w100['lbfgs_l2']:.4f}%")
print(f"Reference: ~0.86% final  (should rescue)")

if hist_w100['lbfgs_l2'] > 10.0:
    print("\n[WARN] w=100 did not rescue. LBFGS may not have found a good basin.")
    print("       The per-soliton comparison will show two failing models.")
    print("       Consider increasing adam_iter further or re-running with a different seed.")
else:
    print("\n[OK] w=100 rescued successfully.")


### Step 3 — Training summary

In [ ]:
print("=" * 55)
print(f"  {'config':<12} {'Adam L2':>10} {'Final L2':>10}")
print("-" * 55)
print(f"  {'w=50':<12} {hist_w50['adam_l2']:>9.4f}% {hist_w50['lbfgs_l2']:>9.4f}%")
print(f"  {'w=100':<12} {hist_w100['adam_l2']:>9.4f}% {hist_w100['lbfgs_l2']:>9.4f}%")
print("=" * 55)
print(f"  Reference: w=50 -> ~42.15%,  w=100 -> ~0.86%")


### Step 4 — Per-soliton windowed L² error

For soliton *i* with speed cᵢ and initial position x0ᵢ, its centre at time t
is xᵢ(t) = x0ᵢ + cᵢ·t. We integrate the squared error inside a ±3-unit window
around that moving centre, normalised by ∫u_exact² in the same window.

In [ ]:
def per_soliton_errors(model, cfg, half_width=3.0, n_x=2000, n_t=200):
    """Compute local L2 error per soliton in a +-half_width window around
    each soliton's moving centre, integrated over all t snapshots."""
    speeds = cfg["speeds"]
    x0s    = cfg["x0s"]
    xL, xR = cfg["domain"]
    T       = cfg["T_END"]

    x  = np.linspace(xL, xR, n_x)
    t  = np.linspace(0.0, T,  n_t)
    dx = x[1] - x[0]

    X, Tg = np.meshgrid(x, t, indexing="ij")   # (n_x, n_t)

    xs_t = torch.tensor(X.flatten(),  dtype=torch.float32, device=K.DEVICE).reshape(-1, 1)
    ts_t = torch.tensor(Tg.flatten(), dtype=torch.float32, device=K.DEVICE).reshape(-1, 1)

    with torch.no_grad():
        u_pred = model(xs_t, ts_t).cpu().numpy().reshape(n_x, n_t)

    u_exact = K.multisoliton_np(X, Tg, speeds, x0s)   # (n_x, n_t)
    err2    = (u_pred - u_exact) ** 2

    rows = []
    for i, (c, x0) in enumerate(zip(speeds, x0s)):
        x_centre = x0 + c * t          # soliton centre at each t,  shape (n_t,)
        num = 0.0
        den = 0.0
        for j in range(n_t):
            lo   = x_centre[j] - half_width
            hi   = x_centre[j] + half_width
            mask = (x >= lo) & (x <= hi)
            if mask.sum() < 2:          # need at least 2 points to integrate
                continue
            num += np.trapezoid(err2[mask, j],  dx=dx)
            den += np.trapezoid(u_exact[mask, j] ** 2, dx=dx)
        l2_local = 100.0 * np.sqrt(num / den) if den > 1e-30 else float("nan")
        rows.append(dict(soliton=i + 1, c=c, x0=x0, local_L2_pct=l2_local))

    return pd.DataFrame(rows)

print("Computing per-soliton errors for w=50 ...")
df_w50  = per_soliton_errors(model_w50,  cfg)
df_w50["model"] = "w50_failed"
print("Done.")

print("Computing per-soliton errors for w=100 ...")
df_w100 = per_soliton_errors(model_w100, cfg)
df_w100["model"] = "w100_rescued"
print("Done.")

print("\n--- w=50 ---")
print(df_w50[["soliton","c","x0","local_L2_pct"]].to_string(index=False,
      float_format=lambda v: f"{v:.4f}"))
print("\n--- w=100 ---")
print(df_w100[["soliton","c","x0","local_L2_pct"]].to_string(index=False,
      float_format=lambda v: f"{v:.4f}"))

pd.concat([df_w50, df_w100], ignore_index=True).to_csv(
    OUT / "per_soliton_errors.csv", index=False)
print(f"\nsaved per_soliton_errors.csv")


### Step 5 — Bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, df, title, glob_l2 in [
    (axes[0], df_w50,  f"w=50  (failed,   global L2={hist_w50['lbfgs_l2']:.2f}%)",
               hist_w50["lbfgs_l2"]),
    (axes[1], df_w100, f"w=100 (rescued?, global L2={hist_w100['lbfgs_l2']:.2f}%)",
               hist_w100["lbfgs_l2"]),
]:
    ax.bar(df["soliton"], df["local_L2_pct"], color="steelblue", edgecolor="k", width=0.7)
    ax.axhline(glob_l2, ls="--", color="red", lw=1.5, label=f"global L2 = {glob_l2:.2f}%")
    ax.set_xlabel("soliton index  (1 = fastest / leading)")
    ax.set_ylabel("local L2 error (%)")
    ax.set_title(title, fontsize=10)
    ax.set_xticks(df["soliton"])
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis="y")

fig.suptitle("Per-soliton L2 error — N=12, ±3-unit window around each moving centre",
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT / "per_soliton_bars.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"saved per_soliton_bars.png")


### Step 6 — Annotated heatmaps

In [ ]:
def annotated_heatmap(model, cfg, save_path, title):
    X, Tg, u_pred, u_exact = K.predict_grid(model, cfg, n_x=600, n_t=200)
    err = np.abs(u_pred - u_exact)

    fig, ax = plt.subplots(figsize=(13, 5))
    im = ax.imshow(
        err.T, origin="lower", aspect="auto",
        extent=[cfg["domain"][0], cfg["domain"][1], 0.0, cfg["T_END"]],
        cmap="hot",
    )
    # overlay soliton tracks
    t_track = np.linspace(0, cfg["T_END"], 60)
    for i, (c, x0) in enumerate(zip(cfg["speeds"], cfg["x0s"])):
        xc = x0 + c * t_track
        ax.plot(xc, t_track, color="cyan", lw=0.7, alpha=0.8)
        # label at t=T_END
        ax.text(xc[-1] + 0.5, t_track[-1] - 0.15, str(i + 1),
                color="cyan", fontsize=7, ha="left", va="top")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, label="|u_pred - u_exact|")
    fig.tight_layout()
    fig.savefig(save_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"saved {save_path.name}  (max err = {err.max():.4f})")

annotated_heatmap(
    model_w50, cfg,
    OUT / "heatmap_N12_w50_annotated.png",
    f"N=12 w=50 — failed (global L2 = {hist_w50['lbfgs_l2']:.2f}%) — cyan = soliton tracks",
)

annotated_heatmap(
    model_w100, cfg,
    OUT / "heatmap_N12_w100_annotated.png",
    f"N=12 w=100 — rescued? (global L2 = {hist_w100['lbfgs_l2']:.2f}%) — cyan = soliton tracks",
)


### Step 7 — Interpretation

In [ ]:
def describe(df, label):
    arr = df["local_L2_pct"].dropna().values
    if len(arr) == 0:
        print(f"\n{label}: no valid data"); return None
    cv = arr.std() / arr.mean() if arr.mean() > 0 else 0
    worst_idx = df["local_L2_pct"].idxmax()
    best_idx  = df["local_L2_pct"].idxmin()
    print(f"\n{label}")
    print(f"  mean = {arr.mean():.3f}%   std = {arr.std():.3f}%   CV = {cv:.3f}")
    print(f"  worst: soliton {int(df.loc[worst_idx,'soliton'])}  "
          f"(c={df.loc[worst_idx,'c']:.2f})  {df.loc[worst_idx,'local_L2_pct']:.3f}%")
    print(f"  best:  soliton {int(df.loc[best_idx,'soliton'])}  "
          f"(c={df.loc[best_idx,'c']:.2f})  {df.loc[best_idx,'local_L2_pct']:.3f}%")
    verdict = "highly non-uniform -> partial basin (optimisation failure)" if cv > 0.2               else "uniform -> capacity limit"
    print(f"  -> {verdict}")
    return cv

cv_w50  = describe(df_w50,  "w=50  (failed)")
cv_w100 = describe(df_w100, "w=100 (rescued?)")

summary = dict(
    w50  = dict(global_L2=hist_w50["lbfgs_l2"],  cv=cv_w50,
                per_soliton=df_w50.to_dict("records")),
    w100 = dict(global_L2=hist_w100["lbfgs_l2"], cv=cv_w100,
                per_soliton=df_w100.to_dict("records")),
)
K.save_json(OUT / "per_soliton_summary.json", summary)
print(f"\nsaved per_soliton_summary.json")
print("\nDone. All outputs in", OUT)
